# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the [`mlcroissant`](https://mlcommons.org/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and display their ids and basic info
record_sets = meta.record_sets
print(f"Total record sets found: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record set: {record_set.name}")
    print(f"  @id: {record_set.id}")
    if record_set.description:
        print(f"  Description: {record_set.description}")
    # List fields in this record set
    print("  Fields:")
    for field in record_set.fields:
        print(f"   - {field.name} (@id: {field.id}) [Type: {field.data_type}]")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Below, you can change the `selected_record_set_id` to the `@id` of the record set you'd like to explore (from the overview above).

In [ ]:
# Select the main protein record set for extraction
# Find the @id for the main table:

# For demonstration purposes, select the first non-empty record set
protein_record_set = None
for rs in record_sets:
    if hasattr(rs, 'fields') and rs.fields:
        protein_record_set = rs
        break

if protein_record_set is None:
    raise ValueError("No record set with fields found.")

record_set_id = protein_record_set.id  # Use record_set_id by @id as per instruction
print(f"Selected record set: {protein_record_set.name} @id={record_set_id}")

# Extract all records into a DataFrame
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)

print("Columns in DataFrame:")
print(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, e.g. filtering, normalization, and grouping. Reference all fields by their `@id`.

In [ ]:
import numpy as np

# Let's pick a numeric field, e.g., 'cr:peptideCount' if it exists
numeric_field_id = None
group_field_id = None
for f in protein_record_set.fields:
    if ('int' in f.data_type or 'float' in f.data_type or 'number' in f.data_type or 'Double' in f.data_type or 'Integer' in f.data_type or 'Float' in f.data_type):
        if numeric_field_id is None:
            numeric_field_id = f.id
    if ('group' in f.name.lower() or 'type' in f.name.lower() or 'category' in f.name.lower()):
        group_field_id = f.id
# Fallback: group by a string field if not found
if group_field_id is None:
    for f in protein_record_set.fields:
        if 'str' in f.data_type or 'Text' in f.data_type:
            group_field_id = f.id
            break

print(f"Numeric field selected: {numeric_field_id}")
print(f"Group-by field selected: {group_field_id}")

if numeric_field_id not in df.columns:
    raise KeyError(f"Field {numeric_field_id} not in DataFrame columns.")

# Convert numeric field to float
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove outlier records where the value is extreme or NaN
threshold = np.nanpercentile(df[numeric_field_id], 90)  # Use 90th percentile
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} above {threshold:.2f} (90th percentile):")
print(filtered_df.head())

# Normalize the numeric field for these filtered records
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} (z-score):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the chosen group field, if available
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field and relationship to the group field where possible.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
df[numeric_field_id].hist(bins=30)
plt.title(f"Histogram of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if text or categorical)
if group_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    df.boxplot(column=numeric_field_id, by=group_field_id, grid=False, vert=False)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(numeric_field_id)
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using `mlcroissant`, listing its record sets and fields, extracting data, and applying basic data processing and visualization steps. All data entities (record sets, fields) were referenced by their `@id` for reproducibility and FAIR data handling.

For further analysis, explore additional fields and record sets by updating field or set `@id`s accordingly.

For questions or more advanced use cases, visit the [`mlcroissant` documentation](https://github.com/mlcommons/croissant).